# Capítulo 4: Separando Dados e Instruções

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Playground de Exemplos](#example-playground)

## Configuração

Execute a célula de configuração a seguir para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
%pip install anthropic --quiet

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Import python's built-in regular expression library
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt}
        ],
        system=system
    )
    return message.content[0].text

---

## Lição

Frequentemente, não queremos escrever prompts completos, mas sim **templates de prompts que podem ser modificados posteriormente com dados de entrada adicionais antes de enviar para Claude**. Isso pode ser útil se você quiser que Claude faça a mesma coisa sempre, mas os dados que Claude usa para sua tarefa podem ser diferentes a cada vez. 

Felizmente, podemos fazer isso facilmente **separando o esqueleto fixo do prompt da entrada variável do usuário e, em seguida, substituindo a entrada do usuário no prompt** antes de enviar o prompt completo para Claude. 

Abaixo, percorreremos passo a passo como escrever um template de prompt substituível, bem como como substituir a entrada do usuário.

### Exemplos

Neste primeiro exemplo, estamos pedindo a Claude para agir como um gerador de sons de animais. Note que o prompt completo enviado para Claude é apenas o `PROMPT_TEMPLATE` substituído pela entrada (neste caso, "Vaca"). Observe que a palavra "Vaca" substitui o placeholder `ANIMAL` através de uma f-string quando imprimimos o prompt completo.

**Nota:** Você não precisa chamar sua variável de placeholder de algo específico na prática. Nós a chamamos de `ANIMAL` neste exemplo, mas com a mesma facilidade, poderíamos tê-la chamado de `CREATURE` ou `A` (embora geralmente seja bom ter seus nomes de variáveis específicos e relevantes para que seu template de prompt seja fácil de entender mesmo sem a substituição, apenas para facilitar a análise do usuário). Apenas certifique-se de que o nome que você der à sua variável seja o que você usa na f-string do template de prompt.

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Por que queremos separar e substituir entradas assim? Bem, **templates de prompt simplificam tarefas repetitivas**. Digamos que você construa uma estrutura de prompt que convide usuários terceiros a enviar conteúdo para o prompt (neste caso, o animal cujo som eles querem gerar). Esses usuários terceiros não precisam escrever ou mesmo ver o prompt completo. Tudo o que eles precisam fazer é preencher as variáveis.

Fazemos essa substituição aqui usando variáveis e f-strings, mas você também pode fazê-lo com o método format().

**Nota:** Templates de prompt podem ter quantas variáveis você desejar!

Ao introduzir variáveis de substituição como esta, é muito importante **garantir que Claude saiba onde as variáveis começam e terminam** (vs. instruções ou descrições de tarefas). Vamos olhar um exemplo onde não há separação entre as instruções e a variável de substituição.

Para nossos olhos humanos, é muito claro onde a variável começa e termina no template de prompt abaixo. No entanto, no prompt totalmente substituído, essa delimitação se torna pouco clara.

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Aqui, **Claude pensa que "Yo Claude" faz parte do email que ele deve reescrever**! Você pode perceber porque ele começa sua reescrita com "Dear Claude". Para o olho humano, está claro, particularmente no template de prompt onde o email começa e termina, mas se torna muito menos claro no prompt após a substituição.

Como resolvemos isso? **Envolva a entrada em tags XML**! Fizemos isso abaixo e, como você pode ver, não há mais "Dear Claude" na saída.

[Tags XML](https://docs.anthropic.com/claude/docs/use-xml-tags) são tags de colchetes angulares como `<tag></tag>`. Elas vêm em pares e consistem em uma tag de abertura, como `<tag>`, e uma tag de fechamento marcada por `/`, como `</tag>`. Tags XML são usadas para envolver conteúdo, assim: `<tag>content</tag>`.

**Nota:** Embora Claude possa reconhecer e trabalhar com uma ampla gama de separadores e delimitadores, recomendamos que você **use especificamente tags XML como separadores** para Claude, pois Claude foi treinado especificamente para reconhecer tags XML como um mecanismo de organização de prompts. Fora da chamada de função, **não há tags XML de molho especial nas quais Claude foi treinado que você deveria usar para maximizar seu desempenho**. Fizemos Claude intencionalmente muito maleável e personalizável dessa forma.

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Vamos ver outro exemplo de como tags XML podem nos ajudar. 

No prompt a seguir, **Claude interpreta incorretamente qual parte do prompt é a instrução vs. a entrada**. Ele considera incorretamente `Cada uma é sobre um animal, como coelhos` como parte da lista devido à formatação, quando o usuário (aquele que preenche a variável `SENTENCES`) presumivelmente não queria isso.

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Para corrigir isso, só precisamos **envolver as frases de entrada do usuário em tags XML**. Isso mostra a Claude onde os dados de entrada começam e terminam, apesar do hífen enganoso antes de `Cada uma é sobre um animal, como coelhos.`

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**Nota:** Na versão incorreta do prompt "Cada uma é sobre um animal", tivemos que incluir o hífen para fazer Claude responder incorretamente da maneira que queríamos para este exemplo. Esta é uma lição importante sobre prompting: **pequenos detalhes importam**! Sempre vale a pena **verificar seus prompts em busca de erros de digitação e erros gramaticais**. Claude é sensível a padrões (em seus primeiros anos, antes do ajuste fino, era uma ferramenta bruta de previsão de texto), e é mais provável que cometa erros quando você comete erros, mais inteligente quando você parece inteligente, mais bobo quando você parece bobo, e assim por diante.

Se você quiser experimentar com os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar o [**Playground de Exemplos**](#example-playground).

---

## Exercícios
- [Exercício 4.1 - Tópico de Haiku](#exercise-41---haiku-topic)
- [Exercício 4.2 - Pergunta sobre Cães com Erros de Digitação](#exercise-42---dog-question-with-typos)
- [Exercício 4.3 - Pergunta sobre Cães Parte 2](#exercise-42---dog-question-part-2)

### Exercício 4.1 - Tópico de Haiku
Modifique o `PROMPT` para que seja um template que receberá uma variável chamada `TOPIC` e produzirá um haiku sobre o tópico. Este exercício serve apenas para testar sua compreensão da estrutura de template de variáveis com f-strings.

In [ ]:
# Variable content
TOPIC = "Pigs"

# Prompt template with a placeholder for the variable content
PROMPT = f""

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
print(hints.exercise_4_1_hint)

### Exercício 4.2 - Pergunta sobre Cães com Erros de Digitação
Corrija o `PROMPT` adicionando tags XML para que Claude produza a resposta certa. 

Tente não mudar mais nada sobre o prompt. A escrita confusa e cheia de erros é intencional, para que você possa ver como Claude reage a tais erros.

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
print(hints.exercise_4_2_hint)

### Exercício 4.3 - Pergunta sobre Cães Parte 2
Corrija o `PROMPT` **SEM** adicionar tags XML. Em vez disso, remova apenas uma ou duas palavras do prompt.

Assim como nos exercícios acima, tente não mudar mais nada sobre o prompt. Isso mostrará que tipo de linguagem Claude pode analisar e entender.

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
print(hints.exercise_4_3_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para avançar para o próximo capítulo. Bons prompts!

---

## Playground de Exemplos

Esta é uma área para você experimentar livremente com os exemplos de prompts mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas de Claude.

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))